In [3]:
import asyncio
import json
import logging
import math
import random
import re
import uuid

import requests
from playwright.async_api import async_playwright

logging.basicConfig(level=logging.NOTSET)
handle = "tesco_api"
logger = logging.getLogger(handle)

user_agent_strings = [
    "Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/45.0.2454.85 Safari/537.36",
    "Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/45.0.2454.85 Safari/537.36",
]

with open("/Users/brianbarry/code/supermarket_scraping/tesco/graphql_query.txt") as file:
    TESCO_GRAPHQL_QUERY = file.read().rstrip()

In [2]:
# Loop through product category pages to extract product hrefs, tpnc product ids can then be extracted from these
from pathlib import Path

PROGRESS_FILE = "ie_tesco_hrefs.json"

tesco_ie_product_categories = [
    "fresh-food",
    "bakery",
    "frozen-food",
    "treats-and-snacks",
    "food-cupboard",
    "drinks",
    "baby-and-toddler",
    "health-and-beauty",
    "pets",
    "household",
    "home-and-living",
]


def load_progress():
    if Path(PROGRESS_FILE).exists():
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {"completed_categories": [], "current_category": None, "current_page": 1, "hrefs": []}


def save_progress(progress):
    with open(PROGRESS_FILE, "w") as f:
        json.dump(progress, f, indent=2)


async def extract_page_hrefs(page):
    await page.locator(".WL_DZkV_Rvg0WJi").last.wait_for()
    hrefs = await page.locator(".WL_DZkV_Rvg0WJi a").evaluate_all(
        "els => [...new Set(els.map(el => el.href).filter(h => h.includes('/products/')))]"
    )
    return hrefs


async def extract_category_hrefs(page, tesco_category, progress, start_page):
    initial_url = f"https://www.tesco.ie/groceries/en-IE/shop/{tesco_category}/all?sortBy=relevance&page={start_page}&count=48#top"
    await page.goto(initial_url, timeout=0, wait_until="load")
    accept_button = page.get_by_text("Accept all")
    if await accept_button.is_visible():
        await accept_button.click()
    pagination_string = await page.get_by_test_id("pagination-result-count").text_content()
    total_products = int(
        re.findall(r"\d+", pagination_string.replace(",", ""))[-1]
    )  # Last number is total
    max_pages = math.ceil(total_products / 48)
    logger.info(f" Max Pages: {max_pages}")
    page_hrefs = await extract_page_hrefs(page)
    progress["hrefs"].extend(page_hrefs)
    progress["current_page"] = start_page
    save_progress(progress)
    logger.info(
        f"{tesco_category} - Page {start_page}/{max_pages} - Total hrefs: {len(progress['hrefs'])}"
    )
    for i in range(start_page + 1, max_pages + 1):
        await asyncio.sleep(random.uniform(2, 5))
        await page.goto(
            f"https://www.tesco.ie/groceries/en-IE/shop/{tesco_category}/all?sortBy=relevance&page={i}&count=48#top",
            timeout=0,
            wait_until="domcontentloaded",
        )
        page_hrefs = await extract_page_hrefs(page)
        progress["hrefs"].extend(page_hrefs)
        progress["current_page"] = i
        save_progress(progress)
        logger.info(
            f"{tesco_category} - Page {i}/{max_pages} - Total hrefs: {len(progress['hrefs'])}"
        )


async def scrape_all_categories(categories):
    progress = load_progress()

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--disable-dev-shm-usage",
                "--no-sandbox",
            ],
        )

        context = await browser.new_context(
            user_agent=random.choice(user_agent_strings),
            viewport={"width": 1920, "height": 1080},
        )

        await context.add_init_script(
            "Object.defineProperty(navigator, 'webdriver', { get: () => undefined })"
        )
        page = await context.new_page()
        all_hrefs = []
        for category in categories:
            if category in progress["completed_categories"]:
                logger.info(f"Skipping completed: {category}")
                continue
            # Determine start page
            if progress["current_category"] == category:
                start_page = progress["current_page"] + 1
            elif progress["current_category"] == category:
                start_page = progress["current_page"]
            else:
                start_page = 1
                progress["current_category"] = category
            # Reuse same page/context for all categories
            logger.info(f"Scraping Category: {category}")
            await extract_category_hrefs(page, category, progress, start_page)
            progress["completed_categories"].append(category)
            progress["current_category"] = None
            progress["current_page"] = 0
            save_progress(progress)
            await asyncio.sleep(random.uniform(5, 10))
        await browser.close()
    return all_hrefs


final = await scrape_all_categories(tesco_ie_product_categories)

INFO:tesco_api:Skipping completed: fresh-food
INFO:tesco_api:Skipping completed: bakery
INFO:tesco_api:Skipping completed: frozen-food
INFO:tesco_api:Skipping completed: treats-and-snacks
INFO:tesco_api:Skipping completed: food-cupboard
INFO:tesco_api:Skipping completed: drinks
INFO:tesco_api:Skipping completed: baby-and-toddler
INFO:tesco_api:Scraping Category: health-and-beauty
INFO:tesco_api: Max Pages: 54
INFO:tesco_api:health-and-beauty - Page 53/54 - Total hrefs: 19848
INFO:tesco_api:health-and-beauty - Page 54/54 - Total hrefs: 19861
INFO:tesco_api:Scraping Category: pets
INFO:tesco_api: Max Pages: 12
INFO:tesco_api:pets - Page 1/12 - Total hrefs: 19909
INFO:tesco_api:pets - Page 2/12 - Total hrefs: 19957
INFO:tesco_api:pets - Page 3/12 - Total hrefs: 20005
INFO:tesco_api:pets - Page 4/12 - Total hrefs: 20053
INFO:tesco_api:pets - Page 5/12 - Total hrefs: 20101
INFO:tesco_api:pets - Page 6/12 - Total hrefs: 20149
INFO:tesco_api:pets - Page 7/12 - Total hrefs: 20197
INFO:tesco_ap

In [ ]:
# Get Tesco sitemap XML to extract all UK product ids
async def fetch_sitemap():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False, args=["--disable-blink-features=AutomationControlled"]
        )

        context = await browser.new_context(
            user_agent=random.choice(user_agent_strings),
            viewport={"width": 1920, "height": 1080},
        )

        await context.add_init_script(
            "Object.defineProperty(navigator, 'webdriver', { get: () => undefined })"
        )
        content = ""
        page = await browser.new_page()
        for i in range(1, 10):
            await page.goto(f"https://www.tesco.com/sitemaps/en-GB/groceries/products-{i}.xml")

            # Wait for and click the Cloudflare checkbox
            # It's usually in an iframe

            # checkbox = page.locator("input[type='checkbox']")

            # # Wait for checkbox to be visible then click
            # await checkbox.wait_for(timeout=50000)
            # await checkbox.click()

            # Wait for challenge to complete and page to load
            await page.wait_for_timeout(5000)

            content += await page.content()
        await browser.close()
        return content


tesco_xml = await fetch_sitemap()
tesco_products = re.findall(
    r"<loc>(https://www.tesco.com/groceries/en-GB/products/[^<]+)</loc>", tesco_xml
)
tesco_ids = re.findall(r"/products/(\d+)</loc>", tesco_xml)

In [6]:
with open("/Users/brianbarry/Documents/supermarket_scraping/notebooks/ie_tesco_hrefs.json") as f:
    tesco_ie_ids = json.load(f)

In [7]:
with open("/Users/brianbarry/Documents/supermarket_scraping/tesco/ie_tesco_ids.csv") as f:
    next(f)  # Skip header
    TESCO_IDS = f.read().splitlines()

In [8]:
len(TESCO_IDS)

19340

In [6]:
tesco_ie_tpncs = list(set(TESCO_IDS))
with open("/Users/brianbarry/Documents/supermarket_scraping/tesco/ie_tesco_ids.csv", "w", newline="") as f:
    f.write("tpnc\n")
    f.write("\n".join(tesco_ie_tpncs))

In [8]:
import json

session = requests.Session()

API_KEY = "TvOSZJHlEk0pjniDGQFAc9Q59WGAR4dA"
tpnc = "307337662"


def get_headers():
    trace_id = str(uuid.uuid4())
    return {
        "accept": "application/json",
        "content-type": "application/json",
        "language": "en-IE",
        "origin": "https://www.tesco.ie",
        "referer": "https://www.tesco.ie/groceries/en-IE/products/250005606",
        "region": "IE",
        "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
        "x-apikey": API_KEY,
        "traceid": f"{trace_id}:{uuid.uuid4()}",
        "trkid": trace_id,
    }


def build_payload(tpnc: str) -> list:
    return [
        {
            "operationName": "GetProduct",
            "variables": {
                "includeVariations": True,
                "includeFulfilment": True,
                "includeMatchingProducts": True,
                "tpnc": tpnc,
                "skipReviews": False,  # Skip reviews to speed up response
                "offset": 0,
                "count": 10,
                "sellersType": "ALL",
                "sellerTypeForVariations": "TOP",
                "productReviewsNodeMaxTimeout": 380,
            },
            "extensions": {"mfeName": "mfe-pdp"},
            "query": TESCO_GRAPHQL_QUERY,
        }
    ]


# First visit the main site to get cookies
session.get("https://www.tesco.ie", headers=get_headers())

# Then try the API
response = session.post("https://xapi.tesco.com/", headers=get_headers(), json=build_payload(tpnc))

print(response.status_code)
tesco_product_info = json.loads(response.text)[0]["data"]

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): www.tesco.ie:443
DEBUG:urllib3.connectionpool:https://www.tesco.ie:443 "GET / HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): xapi.tesco.com:443
DEBUG:urllib3.connectionpool:https://xapi.tesco.com:443 "POST / HTTP/1.1" 200 3567


200


In [9]:
tesco_product_info['product']

{'id': '307337662',
 'baseProductId': '88503803',
 'isRestrictedOrderAmendment': None,
 'gtin': '00282700000000',
 'tpnb': '88503803',
 'tpnc': '307337662',
 'title': 'Tesco Super Seeded Bloomer',
 'description': ['Brown seeded loaf made with mixed seeds and grains.'],
 'brandName': 'TESCO',
 'isInFavourites': None,
 'defaultImageUrl': 'https://digitalcontent.api.tesco.com/v2/media/ghs/719382be-8695-47f3-a033-e9236e9d3681/c6117842-6867-4829-b29e-5ff9aa597ee7_2006996006.jpeg?h=225&w=225',
 'superDepartmentName': "Valentine's Day",
 'superDepartmentId': 'b;VmFsZW50aW5lJ3MlMjBEYXk=',
 'departmentName': "Valentine's Breakfast",
 'departmentId': 'b;VmFsZW50aW5lJ3MlMjBEYXklN0NWYWxlbnRpbmUncyUyMEJyZWFrZmFzdA==',
 'aisleName': "Valentine's Breakfast",
 'aisleId': 'b;VmFsZW50aW5lJ3MlMjBEYXklN0NWYWxlbnRpbmUncyUyMEJyZWFrZmFzdCU3Q1ZhbGVudGluZSdzJTIwQnJlYWtmYXN0',
 'seller': None,
 'shelfName': 'Valentines Breakfast',
 'displayType': 'Quantity',
 'shelfId': 'b;VmFsZW50aW5lJ3MlMjBEYXklN0NWYWxlbnRpbm

In [ ]:
import polars as pl

# Login using e.g. `huggingface-cli login` to access this dataset
df = pl.read_parquet(
    "hf://datasets/Rif-SQL/time-series-uk-retail-supermarket-price-data/base_retail_gb_snappy.parquet"
)
df.filter(
    (pl.col("supermarket_name") == "Tesco") & (pl.col("product_name") == "Tesco Celery")
).sort(pl.col("capture_date"))